<a href="https://colab.research.google.com/github/Hesctory/1st-Activity-Introduction-to-Web-Development/blob/main/pauNaJaca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install torch torchvision

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).





---


---



---
**Dataset WISDM**



---



---



---





In [25]:
# Create time windows

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
import re

# Function to create time windows
def create_time_windows(data, labels, window_size):
    X = []
    y = []

    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])  # Select window of data
        y.append(labels[i + (window_size-1)])  # Label is from the last element of the window

    return np.array(X), np.array(y)

# Updated data loading function
def load_data(file_path):
    features = []
    labels = []

    with open(file_path, 'r') as f:
        for line in f:
            # Use regular expression to match the pattern {{feature_vector}, label}
            match = re.match(r"\{\{([0-9.,-]+)\},\s*(\d+)\}", line.strip())

            if match:
                # Extract feature vector and label
                feature_str = match.group(1)  # The feature string "8.24,-2.11,3.87"
                label = int(match.group(2))  # The label "4"

                # Convert the feature string to a list of floats
                feature_vector = list(map(float, feature_str.split(',')))

                features.append(feature_vector)
                labels.append(label)

    return np.array(features), np.array(labels)

# Load train and test data
train_data_file = '/content/drive/MyDrive/train.dat'  # Adjust path to your file
test_data_file = '/content/drive/MyDrive/test.dat'  # Adjust path to your file

# Define the window size
window_size = 10

# Create time windows
X_train, y_train = load_data(train_data_file)
X_train, y_train = create_time_windows(X_train, y_train, window_size)

X_test, y_test = load_data(test_data_file)
X_test, y_test = create_time_windows(X_test, y_test, window_size)

# Print the first feature vector and label
print("First feature vector in X_train:")
print(X_train[0])  # First row (first feature vector)
print("First label in y_train:")
print(y_train[0])  # First label

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)  # For classification (long type for labels)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# Print the first feature vector and label
print("First feature vector in X_train:")
print(X_train[0])  # First row (first feature vector)
print("First label in y_train:")
print(y_train[0])  # First label

# Check the shapes of the loaded data
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

First feature vector in X_train:
[[-0.6946377  12.680544    0.50395286]
 [ 5.012288   11.264028    0.95342433]
 [ 4.903325   10.882658   -0.08172209]
 [-0.61291564 18.496431    3.0237172 ]
 [-1.1849703  12.108489    7.205164  ]
 [ 1.3756552  -2.4925237  -6.510526  ]
 [-0.61291564 10.56939     5.706926  ]
 [-0.50395286 13.947236    7.0553403 ]
 [-8.430995   11.413852    5.134871  ]
 [ 0.95342433  1.3756552   1.6480621 ]]
First label in y_train:
0
First feature vector in X_train:
tensor([[-0.6946, 12.6805,  0.5040],
        [ 5.0123, 11.2640,  0.9534],
        [ 4.9033, 10.8827, -0.0817],
        [-0.6129, 18.4964,  3.0237],
        [-1.1850, 12.1085,  7.2052],
        [ 1.3757, -2.4925, -6.5105],
        [-0.6129, 10.5694,  5.7069],
        [-0.5040, 13.9472,  7.0553],
        [-8.4310, 11.4139,  5.1349],
        [ 0.9534,  1.3757,  1.6481]])
First label in y_train:
tensor(0)
X_train shape: torch.Size([738432, 10, 3]), y_train shape: torch.Size([738432])
X_test shape: torch.Size([316462

In [26]:
# Define a model and data normalization according to the train data set

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Assume 'train_data_file' is your original training data
        X_train_temp, y_train_temp = load_data(train_data_file)
        scaler = StandardScaler()
        scaler.fit(X_train_temp)

        # Just to print
        original_means = scaler.mean_
        original_stds = scaler.scale_
        print(f"MEANS = {original_means.tolist()}")
        print(f"STDS = {original_stds.tolist()}")

        # Convert the numpy arrays to torch tensors
        means_tensor = torch.tensor(scaler.mean_, dtype=torch.float32)
        stds_tensor = torch.tensor(scaler.scale_, dtype=torch.float32)

        # Register them as non-trainable buffers
        # They are not considered model parameters to be trained.
        self.register_buffer('means', means_tensor)
        self.register_buffer('stds', stds_tensor)
        # ------------------------------------

        # CNN layers
        self.conv1 = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=3, padding=1)  # input channels = number of features
        #self.conv2 = nn.Conv1d(in_channels=16, out_channels=16, kernel_size=3, padding=1)  # input channels = number of features

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        # Fully connected layers
        # Calculate the input size for the fully connected layer based on the output size of conv1
        self.fc1 = nn.Linear(16 * (X_train.shape[1] // 2), 64)  # Flattened size after pooling
        self.fc2 = nn.Linear(64, 6)  # Assuming 6 classes for classification

    def forward(self, x):
        x = (x - self.means) / self.stds

        x = x.permute(0, 2, 1)  # Add channel dimension (change shape to [batch_size, channels, seq_length])

        #x = torch.relu(self.conv1(x))  # Apply conv1 and pooling
        #x = self.pool(torch.relu(self.conv2(x)))  # Apply conv2 and pooling
        x = self.pool(torch.relu(self.conv1(x)))  # Apply conv1 and pooling

        x = x.view(-1, 16 * (x.shape[2]))  # Flatten for fully connected layer
        x = torch.relu(self.fc1(x))  # Apply first fully connected layer
        x = self.fc2(x)  # Output layer (no activation since we'll apply softmax in loss)
        return x


# Initialize the model
model = SimpleCNN()

# Loss function (cross-entropy for classification)
criterion = nn.CrossEntropyLoss()

# Optimizer (Adam)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# DataLoader for batching
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)





MEANS = [1.0182769835326968, 7.7139884937195875, 0.3890242449048468]
STDS = [6.662064228097328, 6.77728797625153, 4.938553052818819]


In [27]:
print(model)

#print(model.state_dict())


SimpleCNN(
  (conv1): Conv1d(3, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=80, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=6, bias=True)
)


In [28]:
# Training loop
num_epochs = 1  # You can adjust the number of epochs

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()  # Zero the gradients
        outputs = model(inputs)  # Forward pass

        # Calculate loss
        loss = criterion(outputs, labels)
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights

        # Track loss and accuracy
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct_preds += (predicted == labels).sum().item()
        total_preds += labels.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct_preds / total_preds
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

Epoch 1/1, Loss: 0.4209, Accuracy: 0.8547


In [29]:
!pip install -q torchinfo
from torchinfo import summary

summary(model, input_size=(1, X_train.shape[1], 3))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleCNN                                [1, 6]                    --
├─Conv1d: 1-1                            [1, 16, 10]               160
├─MaxPool1d: 1-2                         [1, 16, 5]                --
├─Linear: 1-3                            [1, 64]                   5,184
├─Linear: 1-4                            [1, 6]                    390
Total params: 5,734
Trainable params: 5,734
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.02
Estimated Total Size (MB): 0.02

In [30]:
# Evaluate the model
model.eval()  # Set model to evaluation mode
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    correct_preds = (predicted == y_test).sum().item()
    accuracy = correct_preds / len(y_test)
    print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.7573


In [31]:
# Save the trained model
torch.save(model.state_dict(), 'simple_cnn_model.pth')

# Load the model (if needed)
model = SimpleCNN()
model.load_state_dict(torch.load('simple_cnn_model.pth'))
model.eval()

MEANS = [1.0182769835326968, 7.7139884937195875, 0.3890242449048468]
STDS = [6.662064228097328, 6.77728797625153, 4.938553052818819]


SimpleCNN(
  (conv1): Conv1d(3, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=80, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=6, bias=True)
)

In [32]:
!pip install onnxscript onnx

In [33]:
# --- Apply pruning and export whole model ---

import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- Assume your SimpleCNN and data (X_train, y_train, X_test, y_test, window_size) are already defined --- #

# Initialize the model and load trained weights
model = SimpleCNN()
model.load_state_dict(torch.load('simple_cnn_model.pth'))
model.eval()  # Set to evaluation mode

# --- Optional: Evaluate original model accuracy ---
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)
    print(f"Original Test Accuracy: {accuracy:.4f}")

# --- Apply pruning ---
prune_amount = 0.3  # prune 30% of the weights

# Layer-wise pruning
prune.l1_unstructured(model.conv1, name='weight', amount=prune_amount)
prune.l1_unstructured(model.fc1, name='weight', amount=prune_amount)
prune.l1_unstructured(model.fc2, name='weight', amount=prune_amount)

# --- Optional: Evaluate pruned model (with mask) ---
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)
    print(f"Test Accuracy after pruning (with mask): {accuracy:.4f}")

# --- Make pruning permanent (remove mask) ---
prune.remove(model.conv1, 'weight')
prune.remove(model.fc1, 'weight')
prune.remove(model.fc2, 'weight')

# --- Evaluate pruned model (permanent pruning) ---
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)
    print(f"Test Accuracy after permanent pruning: {accuracy:.4f}")

# --- Save pruned model weights ---
torch.save(model.state_dict(), 'simple_cnn_pruned.pth')
print("✅ Pruned model saved as simple_cnn_pruned.pth")

# --- Export the pruned model to ONNX for Netron ---
dummy_input = torch.randn(1, window_size, 3)  # batch_size, seq_len, features
onnx_file = "simple_cnn_pruned.onnx"

torch.onnx.export(
    model,                # pruned model
    dummy_input,          # dummy input
    onnx_file,            # output file
    export_params=True,   # store trained weights
    opset_version=12,
    input_names=['input'],
    output_names=['output']
)

print(f"✅ Pruned model exported to ONNX: {onnx_file}")


MEANS = [1.0182769835326968, 7.7139884937195875, 0.3890242449048468]
STDS = [6.662064228097328, 6.77728797625153, 4.938553052818819]
Original Test Accuracy: 0.7573
Test Accuracy after pruning (with mask): 0.6618


W1009 14:16:19.342000 10169 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Test Accuracy after permanent pruning: 0.6618
✅ Pruned model saved as simple_cnn_pruned.pth
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Pruned model exported to ONNX: simple_cnn_pruned.onnx


In [34]:

# --- Exports the model structure and weights into a single .onnx file ---

import torch

# Load the saved weights from your training session
model.load_state_dict(torch.load('simple_cnn_model.pth'))
model.eval() # Set the model to evaluation mode

# --- Export to ONNX ---
# Create a dummy input tensor with the correct shape: [batch_size, seq_length, features]
# This shape must match your model's input exactly.
dummy_input = torch.randn(1, window_size, 3)

# Define the output file name
onnx_file = "simple_cnn_wisdm.onnx"

print(f"Exporting model to {onnx_file}...")

# Export the model
torch.onnx.export(model,               # The model to export
                  dummy_input,         # A sample input
                  onnx_file,      # Where to save the model
                  export_params=True,  # Store the trained weights
                  opset_version=12,    # The ONNX version to use
                  input_names=['input'], # The name for the input tensor
                  output_names=['output']) # The name for the output tensor

print("Model has been successfully converted to ONNX format! ✅")

W1009 14:16:22.300000 10169 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Exporting model to simple_cnn_wisdm.onnx...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model has been successfully converted to ONNX format! ✅


In [35]:
!pip install -q netron portpicker

import os
import netron
import portpicker
from google.colab import output
from IPython.display import display, HTML

model_path = "/content/simple_cnn_wisdm.onnx"

assert os.path.exists(model_path), f"Model not found: {model_path}"

port = portpicker.pick_unused_port()

netron.start(
    model_path,
    address=("0.0.0.0", port),
    browse=False
)

output.serve_kernel_port_as_iframe(port, height=800)

<IPython.core.display.Javascript object>

In [36]:
# --- Convert from onnx to tf --- #

!pip install onnx2tf
!pip install onnx_graphsurgeon
!pip install ai-edge-litert
!pip install sng4onnx


!onnx2tf -i simple_cnn_wisdm.onnx -o my_tf_model -fdosm
#!onnx2tf -i simple_cnn_pruned.onnx -o my_tf_model




Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Constant   │ 9              │ 9                │
│ Conv       │ 1              │ 1                │
│ Div        │ 1              │ 1                │
│ Gemm       │ 2              │ 2                │
│ MaxPool    │ 1              │ 1                │
│ Relu       │ 2              │ 2                │
│ Reshape    │ 1              │ 1                │
│ Sub        │ 1              │ 1                │
│ Transpose  │ 1              │ 1                │
│ Model Size │ 32.0KiB        │ 23.7KiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━

In [37]:
# --- Quantize the tf model using TFLM and export as .h --- #
# --- TFLM (TensorFlow Lite for Microcontrollers) --- #

import tensorflow as tf
import numpy as np # Make sure numpy is imported

# This function is your calibration step
def representative_data_gen():
    """
    Feeds sample data from X_train to the converter.
    """
    # Convert X_train tensor to a NumPy array if it isn't already
    # We only need a subset, e.g., the first 100 samples
    x_train_np = X_train.numpy() if isinstance(X_train, torch.Tensor) else X_train
    num_calibration_samples = min(100, x_train_np.shape[0]) # Use up to 100 samples

    print(f"Providing {num_calibration_samples} samples from X_train for calibration...")

    for i in range(num_calibration_samples):
        # 1. Get one window from X_train. Shape: [window_size, features] (e.g., [100, 3])
        sample = x_train_np[i]

        # 2. Transpose to match model input: [features, window_size] (e.g., [3, 100])
        #    This is CRUCIAL because Conv1D expects channels (features) first.
        sample_transposed = np.transpose(sample, (1, 0))

        # 3. Add the batch dimension: [1, features, window_size] (e.g., [1, 3, 100])
        sample_batch = np.expand_dims(sample_transposed, axis=0).astype(np.float32)

        #print(sample_batch)

        # 4. Yield the data in the required list format
        yield [sample_batch]




converter = tf.lite.TFLiteConverter.from_saved_model('my_tf_model')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

with open('simple_cnn_quantized.tflite', 'wb') as f:
#with open('simple_cnn_prunedANDquantized.tflite', 'wb') as f:
    f.write(tflite_model_quant)

print("✅ Success! Your quantized model has been saved as simple_cnn_xxx.tflite")



Providing 100 samples from X_train for calibration...
✅ Success! Your quantized model has been saved as simple_cnn_xxx.tflite


In [38]:
#converts .tflite to hexadecimal (hex) dump

!apt-get update -qq
!apt-get install -y xxd

#!xxd -i model_for_microbit.tflite > model_data.h
!xxd -i simple_cnn_quantized.tflite > model_data.h

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xxd is already the newest version (2:9.1.0016-1ubuntu7.20).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.


In [39]:
#Compare the file sizes

import os

files = {
    "Float32": "/content/my_tf_model/simple_cnn_wisdm_float32.tflite",
    "Quantized": "/content/simple_cnn_quantized.tflite",
    "C header": "/content/model_data.h",
}

sizes = {}

for name, path in files.items():
    if os.path.exists(path):
        sizes[name] = os.path.getsize(path)
        print(f"{name:12}: {sizes[name]:,} bytes ({sizes[name] / 1024:.2f} KB)")
    else:
        print(f"{name}: file not found")


Float32     : 27,132 bytes (26.50 KB)
Quantized   : 12,144 bytes (11.86 KB)
C header    : 74,992 bytes (73.23 KB)


In [40]:
pip install onnx onnxruntime

In [41]:
# --- Test int8 TFLite model properly --- #
# --- Test for pruned, quantized or prunedANDquantized ---#

import tensorflow as tf
import numpy as np

# Load the TFLite model
tflite_model_path = 'simple_cnn_quantized.tflite'
#tflite_model_path = 'simple_cnn_prunedANDquantized.tflite'
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:", input_details)
print("Output details:", output_details)

# Prepare test data as NumPy arrays
X_test_np = X_test.numpy() if isinstance(X_test, torch.Tensor) else X_test
y_test_np = y_test.numpy() if isinstance(y_test, torch.Tensor) else y_test

# Get input quantization parameters
input_scale, input_zero_point = input_details[0]['quantization']
output_scale, output_zero_point = output_details[0]['quantization']

# Accuracy tracking
correct_preds = 0

for i in range(len(X_test_np)):
    # 1. Take one sample
    sample = X_test_np[i]  # shape: [window_size, features]

    # 2. Transpose to [features, window_size]
    sample_transposed = np.transpose(sample, (1, 0))

    # 3. Add batch dimension
    sample_batch = np.expand_dims(sample_transposed, axis=0).astype(np.float32)

    # 4. Quantize input
    input_data = np.round(sample_batch / input_scale + input_zero_point).astype(np.int8)

    # 5. Set input tensor
    interpreter.set_tensor(input_details[0]['index'], input_data)

    # 6. Invoke the interpreter
    interpreter.invoke()

    # 7. Get output and dequantize
    output_data = interpreter.get_tensor(output_details[0]['index'])
    output_data = output_scale * (output_data.astype(np.float32) - output_zero_point)

    # 8. Predicted class
    predicted_class = np.argmax(output_data)

    # 9. Compare with true label
    if predicted_class == y_test_np[i]:
        correct_preds += 1

# Calculate accuracy
accuracy = correct_preds / len(X_test_np)
print(f"Test Accuracy of TFLite int8 model: {accuracy:.4f}")


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Input details: [{'name': 'serving_default_input:0', 'index': 0, 'shape': array([ 1,  3, 10], dtype=int32), 'shape_signature': array([ 1,  3, 10], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.1167076826095581, -41), 'quantization_parameters': {'scales': array([0.11670768], dtype=float32), 'zero_points': array([-41], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'PartitionedCall:0', 'index': 25, 'shape': array([1, 6], dtype=int32), 'shape_signature': array([1, 6], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.28932803869247437, 82), 'quantization_parameters': {'scales': array([0.28932804], dtype=float32), 'zero_points': array([82], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]
Test Accuracy of TFLite int8 model: 0.7468






---


---



---
**Dataset PAMAP2**



---



---

In [42]:
#remove columns heartrate and temp of the three IMUs

import pandas as pd

# Define the file path
train_data_file = '/content/drive/MyDrive/pamap2.csv'

# Load the CSV file into a DataFrame
df = pd.read_csv(train_data_file)

# Remove columns
df = df.drop(df.columns[2], axis=1)
df = df.drop(df.columns[2], axis=1)
df = df.drop(df.columns[11], axis=1)
df = df.drop(df.columns[20], axis=1)


# Save the updated DataFrame to a new CSV file
output_file = '/content/drive/MyDrive/pamap2_columns_removed.csv'
df.to_csv(output_file, index=False)

# Print a message to confirm that the file was saved
print(f"Updated CSV saved to: {output_file}")


Updated CSV saved to: /content/drive/MyDrive/pamap2_columns_removed.csv


In [43]:
# Replace the activity names in the first column with their corresponding IDs

import pandas as pd

# Define the file path for the modified CSV
modified_train_data_file = '/content/drive/MyDrive/pamap2_columns_removed.csv'

# Load the modified CSV into a DataFrame without assuming a header
df = pd.read_csv(modified_train_data_file, header=None)

# Print the first few rows to check the content
print("Original DataFrame first few rows:")
print(df.head())

# Create a dictionary to map activity names to activity IDs
activity_map = {
    'lying': 1,
    'sitting': 2,
    'standing': 3,
    'walking': 4,
    'running': 5,
    'cycling': 6,
    'nordic_walking': 7,
    'watching_TV': 9,
    'computer_work': 10,
    'car_driving': 11,
    'ascending_stairs': 12,
    'descending_stairs': 13,
    'vacuum_cleaning': 16,
    'ironing': 17,
    'folding_laundry': 18,
    'house_cleaning': 19,
    'playing_soccer': 20,
    'rope_jumping': 24,
    'other': 0
}

# Replace the activity names in the first column with their corresponding IDs
df.iloc[:, 0] = df.iloc[:, 0].map(activity_map)

# Print the first few rows after the transformation to ensure it's working
print("\nDataFrame after replacing activity names with IDs:")
print(df.head())

# Save the updated DataFrame to a new CSV file
output_file = '/content/drive/MyDrive/pamap2_with_activityIDs.csv'
df.to_csv(output_file, index=False, header=False)

# Print a confirmation message
print(f"Updated CSV with activity IDs saved to: {output_file}")


Original DataFrame first few rows:
      0    1        2        3        4         5         6         7   \
0  lying  1.0  2.21530  8.27915  5.58753 -0.004750  0.037579 -0.011145   
1  lying  1.0  2.29196  7.67288  5.74467 -0.171710  0.025479 -0.009538   
2  lying  1.0  2.29090  7.14240  5.82342 -0.238241  0.011214  0.000831   
3  lying  1.0  2.21800  7.14365  5.89930 -0.192912  0.019053  0.013374   
4  lying  1.0  2.30106  7.25857  6.09259 -0.069961 -0.018328  0.004582   

        8        9   ...       19       20       21        22        23  \
0  8.93200 -67.9326  ...  44.2728  9.73855 -1.84761  0.095156  0.002908   
1  9.58300 -67.9584  ...  43.5427  9.69762 -1.88438 -0.020804  0.020882   
2  9.05516 -67.4017  ...  44.0259  9.69633 -1.92203 -0.059173 -0.035392   
3  9.92698 -67.4387  ...  43.6570  9.66370 -1.84714  0.094385 -0.032514   
4  9.15626 -67.1825  ...  42.9228  9.77578 -1.88582  0.095775  0.001351   

         24        25       26       27       28  
0 -0.027714  0.001

In [45]:
#Split CSV in train (excluding user 5) and test (only user 5) csv files

import pandas as pd

# Define the file path for the CSV file containing activity IDs
train_data_file_with_ids = '/content/drive/MyDrive/pamap2_with_activityIDs.csv'

# Load the CSV file into a DataFrame without assuming a header
df = pd.read_csv(train_data_file_with_ids, header=None)

# Print the first few rows to check the content
print("Original DataFrame first few rows:")
print(df.head())

# Filter rows where the user ID is 5 and save them as test_pamap2.csv
test_df = df[df.iloc[:, 1] == 5]
test_df.to_csv('/content/drive/MyDrive/test_pamap2.csv', index=False, header=False)

# Filter rows where the user ID is not 5 (excluding) and save them as train_pamap2.csv
train_df = df[df.iloc[:, 1] != 5]
train_df.to_csv('/content/drive/MyDrive/train_pamap2.csv', index=False, header=False)

# Print a confirmation message
print("Data has been split into train_pamap2.csv (excluding user ID 5) and test_pamap2.csv (user ID 5 only).")


Original DataFrame first few rows:
   0    1        2        3        4         5         6         7        8   \
0   1  1.0  2.21530  8.27915  5.58753 -0.004750  0.037579 -0.011145  8.93200   
1   1  1.0  2.29196  7.67288  5.74467 -0.171710  0.025479 -0.009538  9.58300   
2   1  1.0  2.29090  7.14240  5.82342 -0.238241  0.011214  0.000831  9.05516   
3   1  1.0  2.21800  7.14365  5.89930 -0.192912  0.019053  0.013374  9.92698   
4   1  1.0  2.30106  7.25857  6.09259 -0.069961 -0.018328  0.004582  9.15626   

        9   ...       19       20       21        22        23        24  \
0 -67.9326  ...  44.2728  9.73855 -1.84761  0.095156  0.002908 -0.027714   
1 -67.9584  ...  43.5427  9.69762 -1.88438 -0.020804  0.020882  0.000945   
2 -67.4017  ...  44.0259  9.69633 -1.92203 -0.059173 -0.035392 -0.052422   
3 -67.4387  ...  43.6570  9.66370 -1.84714  0.094385 -0.032514 -0.018844   
4 -67.1825  ...  42.9228  9.77578 -1.88582  0.095775  0.001351 -0.048878   

         25       26       

In [46]:
#Convert .csv to .dat

import pandas as pd

# Define file paths
train_file = '/content/drive/MyDrive/train_pamap2.csv'
test_file = '/content/drive/MyDrive/test_pamap2.csv'

# Function to convert rows into the required format and save as .dat
def convert_to_dat(input_file, output_file):
    # Load the CSV file without a header
    df = pd.read_csv(input_file, header=None)

    # Open the output .dat file to write
    with open(output_file, 'w') as file:
        for _, row in df.iterrows():
            # Convert row into the required format: {{column 2, column 3, column 4, ...}, column 0}
            row_data = "{{" + ",".join(map(str, row[2:])) + "}," + str(int(row[0])) + "}\n"
            file.write(row_data)

# Convert train_pamap2.csv to train_pamap2.dat
convert_to_dat(train_file, '/content/drive/MyDrive/train_pamap2.dat')

# Convert test_pamap2.csv to test_pamap2.dat
convert_to_dat(test_file, '/content/drive/MyDrive/test_pamap2.dat')

print("Conversion to .dat files is complete!")


Conversion to .dat files is complete!


In [47]:
# # Create time windows

# import torch
# import torch.nn as nn
# import torch.optim as optim
# import numpy as np
# from torch.utils.data import DataLoader
# import re

# # Function to create time windows
# def create_time_windows(data, labels, window_size):
#     X = []
#     y = []

#     for i in range(len(data) - window_size):
#         X.append(data[i:i + window_size])  # Select window of data
#         y.append(labels[i + (window_size-1)])  # Label is from the last element of the window

#     return np.array(X), np.array(y)

# # Updated data loading function
# def load_data(file_path):
#     features = []
#     labels = []

#     with open(file_path, 'r') as f:
#         for line in f:
#             # Use regular expression to match the pattern {{feature_vector}, label}
#             match = re.match(r"\{\{([0-9.,-]+)\},\s*(\d+)\}", line.strip())

#             if match:
#                 # Extract feature vector and label
#                 feature_str = match.group(1)  # The feature string "8.24,-2.11,3.87"
#                 label = int(match.group(2))  # The label "4"

#                 # Convert the feature string to a list of floats
#                 feature_vector = list(map(float, feature_str.split(',')))

#                 features.append(feature_vector)
#                 labels.append(label)

#     return np.array(features), np.array(labels)

# # Load train and test data
# train_data_file = '/content/drive/MyDrive/train_pamap2.dat'  # Adjust path to your file
# test_data_file = '/content/drive/MyDrive/test_pamap2.dat'  # Adjust path to your file

# print("File loaded:")

# # Define the window size
# window_size = 10

# # Create time windows
# X_train, y_train = load_data(train_data_file)
# X_train, y_train = create_time_windows(X_train, y_train, window_size)

# X_test, y_test = load_data(test_data_file)
# X_test, y_test = create_time_windows(X_test, y_test, window_size)

# print("Time window created:")

# # Print the first feature vector and label
# print("First feature vector in X_train:")
# print(X_train[0])  # First row (first feature vector)
# print("Second feature vector in X_train:")
# print(X_train[1])  # First row (first feature vector)

# print("First label in y_train:")
# print(y_train[0])  # First label
# print("Second label in y_train:")
# print(y_train[1])  # First label

# # Convert to PyTorch tensors
# X_train = torch.tensor(X_train, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.long)  # For classification (long type for labels)

# X_test = torch.tensor(X_test, dtype=torch.float32)
# y_test = torch.tensor(y_test, dtype=torch.long)

# # # Print the first feature vector and label
# # print("First feature vector in X_train:")
# # print(X_train[0])  # First row (first feature vector)
# # print("First label in y_train:")
# # print(y_train[0])  # First label

# # Check the shapes of the loaded data
# print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
# print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

In [48]:
# # Define a model

# class SimpleCNN(nn.Module):
#     def __init__(self):
#         super(SimpleCNN, self).__init__()

#         # CNN layers
#         self.conv1 = nn.Conv1d(in_channels=27, out_channels=27, kernel_size=3, padding=1)  # input channels = number of features
#         self.conv2 = nn.Conv1d(in_channels=27, out_channels=16, kernel_size=3, padding=1)  # input channels = number of features

#         self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

#         # Fully connected layers
#         # Calculate the input size for the fully connected layer based on the output size of conv1
#         self.fc1 = nn.Linear(16 * (X_train.shape[1] // 2), 64)  # Flattened size after pooling
#         self.fc2 = nn.Linear(64, 25)  # Assuming 25 classes for classification

#     def forward(self, x):
#         x = x.permute(0, 2, 1)  # Add channel dimension (change shape to [batch_size, channels, seq_length])

#         x = torch.relu(self.conv1(x))  # Apply conv1 and pooling
#         x = self.pool(torch.relu(self.conv2(x)))  # Apply conv2 and pooling
#         #x = self.pool(torch.relu(self.conv1(x)))  # Apply conv1 and pooling

#         x = x.view(-1, 16 * (x.shape[2]))  # Flatten for fully connected layer
#         x = torch.relu(self.fc1(x))  # Apply first fully connected layer
#         x = self.fc2(x)  # Output layer (no activation since we'll apply softmax in loss)

#         return x


# # Initialize the model
# model = SimpleCNN()

# # Loss function (cross-entropy for classification)
# criterion = nn.CrossEntropyLoss()

# # Optimizer (Adam)
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# # DataLoader for batching
# train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [49]:
# # Evaluate the model
# model.eval()  # Set model to evaluation mode
# with torch.no_grad():
#     outputs = model(X_test)
#     _, predicted = torch.max(outputs, 1)
#     correct_preds = (predicted == y_test).sum().item()
#     accuracy = correct_preds / len(y_test)
#     print(f"Test Accuracy: {accuracy:.4f}")

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import re

# --- 1. NEW DATASET CLASS ---
class PAMAP2Dataset(Dataset):
    def __init__(self, data, labels, window_size):
        # Convert numpy arrays to tensors once
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.window_size = window_size

    def __len__(self):
        # Total windows possible
        return len(self.data) - self.window_size + 1

    def __getitem__(self, idx):
        # Slice the window from the main data array on-the-fly
        x = self.data[idx : idx + self.window_size]
        # Label is from the last element of the window
        y = self.labels[idx + self.window_size - 1]
        return x, y

# --- 2. DATA LOADING FUNCTION (Kept as is) ---
def load_data(file_path):
    features = []
    labels = []
    with open(file_path, 'r') as f:
        for line in f:
            match = re.match(r"\{\{([0-9.,-]+)\},\s*(\d+)\}", line.strip())
            if match:
                feature_str = match.group(1)
                label = int(match.group(2))
                feature_vector = list(map(float, feature_str.split(',')))
                features.append(feature_vector)
                labels.append(label)
    return np.array(features), np.array(labels)

# --- 3. EXECUTION ---

# Load raw flat data
train_data_file = '/content/drive/MyDrive/train_pamap2.dat'  # Adjust path to your file
test_data_file = '/content/drive/MyDrive/test_pamap2.dat'  # Adjust path to your file


raw_X_train, raw_y_train = load_data(train_data_file)
raw_X_test, raw_y_test = load_data(test_data_file)
print(f"Files loaded. Raw shape: {raw_X_train.shape}")

# Define window size
window_size = 100

# Create Dataset objects (No massive arrays created here!)
train_dataset = PAMAP2Dataset(raw_X_train, raw_y_train, window_size)
test_dataset = PAMAP2Dataset(raw_X_test, raw_y_test, window_size)

# Create DataLoaders
# shuffle=True is important for training; it will shuffle the window indices
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Number of training windows: {len(train_dataset)}")
print(f"Number of testing windows: {len(test_dataset)}")

# Verify the first item
first_x, first_y = train_dataset[0]
print("\nFirst window shape:", first_x.shape) # Should be [10, num_features]
print("First window label:", first_y)

Files loaded. Raw shape: (1655794, 27)
Number of training windows: 1655695
Number of testing windows: 270527

First window shape: torch.Size([100, 27])
First window label: tensor(1)


In [52]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Define the model
class SimpleCNN(nn.Module):
    def __init__(self, num_features, window_size, num_classes=25):
        super(SimpleCNN, self).__init__()

        # CNN layers
        # in_channels = number of features (e.g., 27)
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=27, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=27, out_channels=16, kernel_size=3, padding=1)

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        # Calculate the flattened size automatically
        # After one MaxPool1d(kernel_size=2, stride=2), the sequence length is halved
        self.flattened_size = 16 * (window_size // 2)

        # Fully connected layers
        self.fc1 = nn.Linear(self.flattened_size, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # Input x shape: [batch_size, window_size, num_features]
        # Conv1d expects: [batch_size, num_features, window_size]
        x = x.permute(0, 2, 1)

        x = torch.relu(self.conv1(x))
        x = self.pool(torch.relu(self.conv2(x)))

        # Flatten for the fully connected layer
        # x.size(0) is the batch size
        x = x.view(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = self.fc2(x) # No activation here (handled by CrossEntropyLoss)

        return x

# --- Setup for Training ---

# 1. Define your parameters based on your data
num_features = raw_X_train.shape[1]  # Automatically detects 27
window_size = window_size
num_classes = 25

# 2. Initialize the model with parameters
model = SimpleCNN(num_features=num_features, window_size=window_size, num_classes=num_classes)

# 3. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Use the memory-efficient DataLoaders we created earlier
# (Assuming train_dataset was created using the PAMAP2Dataset class)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Model initialized and ready for training!")

Model initialized and ready for training!


In [ ]:
# Training loop

num_epochs = 3  # You can adjust the number of epochs

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()  # Zero the gradients
        outputs = model(inputs)  # Forward pass

        # Calculate loss
        loss = criterion(outputs, labels)
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights

        # Track loss and accuracy
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct_preds += (predicted == labels).sum().item()
        total_preds += labels.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct_preds / total_preds
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

In [ ]:
# --- Correct Evaluation for Memory-Efficient Setup ---

model.eval()  # Set model to evaluation mode (stops dropout)
correct_preds = 0
total_samples = 0

# We use the test_loader to feed data in small batches
with torch.no_grad(): # Disable gradient math to save memory and speed up
    for inputs, labels in test_loader:
        # 1. Forward pass (process one batch of 32 windows)
        outputs = model(inputs)

        # 2. Get the index of the highest value (the predicted activity)
        _, predicted = torch.max(outputs, 1)

        # 3. Compare predictions to actual labels
        correct_preds += (predicted == labels).sum().item()

        # 4. Count how many samples we've checked so far
        total_samples += labels.size(0)

# Calculate final accuracy
accuracy = correct_preds / total_samples
print(f"Test Accuracy: {accuracy:.4f}")